# Engenharia de Features

Este notebook cria uma linha de features para cada partida do Brasileirão
entre 2020 e 2024. Execute antes `python src/data/prepare_matches.py` na raiz
do projeto e depois rode todas as células deste notebook em ordem.

O histórico é ordenado por temporada, data e ID antes de calcular os últimos
cinco jogos. Cada janela usa apenas resultados de partidas anteriores na
mesma temporada. Na primeira partida de cada clube (e no primeiro jogo em
cada mando), as médias ficam vazias porque ainda não há histórico. Esses
valores são mantidos como ausentes na base gerada.

In [1]:
from pathlib import Path
import pandas as pd

caminho_partidas = Path('../data/processed/matches_2020_2024.csv')
if not caminho_partidas.is_file():
    raise FileNotFoundError(
        'Base processada ausente. Na raiz do projeto, execute: '
        'python src/data/prepare_matches.py'
    )

df = pd.read_csv(caminho_partidas, parse_dates=['data'])

In [2]:
df.shape

(1900, 12)

In [3]:
df.head()

,id,temporada,rodada,data,mandante,visitante,gols_mandante,gols_visitante,arena,estado_mandante,estado_visitante,resultado
0,6886,2020,1,2020-08-08,Fortaleza,Athletico-PR,0,2,Arena Castelão,CE,PR,A
1,6887,2020,1,2020-08-08,Coritiba,Internacional,0,1,Couto Pereira,PR,RS,A
2,6888,2020,1,2020-08-08,Sport,Ceara,3,2,Adelmar da Costa Carvalho,PE,CE,H
3,6889,2020,1,2020-08-09,Santos,Bragantino,1,1,Estádio Urbano Caldeira,SP,SP,D
4,6890,2020,1,2020-08-09,Flamengo,Atletico-MG,0,1,Maracanã,RJ,MG,A


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1900 entries, 0 to 1899
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id                1900 non-null   int64         
 1   temporada         1900 non-null   int64         
 2   rodada            1900 non-null   int64         
 3   data              1900 non-null   datetime64[us]
 4   mandante          1900 non-null   str           
 5   visitante         1900 non-null   str           
 6   gols_mandante     1900 non-null   int64         
 7   gols_visitante    1900 non-null   int64         
 8   arena             1900 non-null   str           
 9   estado_mandante   1900 non-null   str           
 10  estado_visitante  1900 non-null   str           
 11  resultado         1900 non-null   str           
dtypes: datetime64[us](1), int64(5), str(6)
memory usage: 178.3 KB


In [5]:
df["temporada"].value_counts().sort_index()

temporada
2020    380
2021    380
2022    380
2023    380
2024    380
Name: count, dtype: int64

In [6]:
jogos_mandante = df[
    [
        "id",
        "temporada",
        "rodada",
        "data",
        "mandante",
        "visitante",
        "gols_mandante",
        "gols_visitante",
        "resultado"
    ]
].copy()

In [7]:
jogos_mandante = jogos_mandante.rename(
    columns={
        "mandante": "clube",
        "visitante": "adversario",
        "gols_mandante": "gols_pro",
        "gols_visitante": "gols_contra"
    }
)

In [8]:
jogos_mandante["mando"] = "casa"

In [9]:
jogos_mandante["pontos"] = 0

jogos_mandante.loc[
    jogos_mandante["resultado"] == "H",
    "pontos"
] = 3

jogos_mandante.loc[
    jogos_mandante["resultado"] == "D",
    "pontos"
] = 1

In [10]:
jogos_visitante = df[
    [
        "id",
        "temporada",
        "rodada",
        "data",
        "visitante",
        "mandante",
        "gols_visitante",
        "gols_mandante",
        "resultado"
    ]
].copy()

In [11]:
jogos_visitante = jogos_visitante.rename(
    columns={
        "visitante": "clube",
        "mandante": "adversario",
        "gols_visitante": "gols_pro",
        "gols_mandante": "gols_contra"
    }
)

In [12]:
jogos_visitante["mando"] = "fora"

In [13]:
jogos_visitante["pontos"] = 0

jogos_visitante.loc[
    jogos_visitante["resultado"] == "A",
    "pontos"
] = 3

jogos_visitante.loc[
    jogos_visitante["resultado"] == "D",
    "pontos"
] = 1

In [14]:
historico_clubes = (
    pd.concat([jogos_mandante, jogos_visitante], ignore_index=True)
    .sort_values(['temporada', 'data', 'id', 'mando'])
    .reset_index(drop=True)
)

In [15]:
historico_clubes.shape

(3800, 11)

In [16]:
historico_clubes["mando"].value_counts()

mando
casa    1900
fora    1900
Name: count, dtype: int64

In [17]:
historico_clubes[
    [
        "data",
        "clube",
        "adversario",
        "mando",
        "gols_pro",
        "gols_contra",
        "pontos"
    ]
].head(10)

,data,clube,adversario,mando,gols_pro,gols_contra,pontos
0,2020-08-08,Fortaleza,Athletico-PR,casa,0,2,0
1,2020-08-08,Athletico-PR,Fortaleza,fora,2,0,3
2,2020-08-08,Coritiba,Internacional,casa,0,1,0
3,2020-08-08,Internacional,Coritiba,fora,1,0,3
4,2020-08-08,Sport,Ceara,casa,3,2,3
5,2020-08-08,Ceara,Sport,fora,2,3,0
6,2020-08-09,Santos,Bragantino,casa,1,1,1
7,2020-08-09,Bragantino,Santos,fora,1,1,1
8,2020-08-09,Flamengo,Atletico-MG,casa,0,1,0
9,2020-08-09,Atletico-MG,Flamengo,fora,1,0,3


In [18]:
historico_clubes["pontos_jogo_anterior"] = (
    historico_clubes
    .groupby(["temporada", "clube"])["pontos"]
    .shift(1)
)

In [19]:
historico_clubes[
    (historico_clubes["clube"] == "Flamengo") &
    (historico_clubes["temporada"] == 2024)
][
    [
        "data",
        "rodada",
        "adversario",
        "pontos",
        "pontos_jogo_anterior"
    ]
].head(10)

,data,rodada,adversario,pontos,pontos_jogo_anterior
3053,2024-04-14,1,Atletico-GO,3,NaN
3074,2024-04-17,2,Sao Paulo,3,3.0
3087,2024-04-21,3,Palmeiras,1,3.0
3102,2024-04-28,4,Botafogo-RJ,0,1.0
3119,2024-05-04,5,Bragantino,1,0.0
3130,2024-05-11,6,Corinthians,3,1.0
3155,2024-06-02,7,Vasco,3,3.0
3184,2024-06-13,8,Gremio,3,3.0
3199,2024-06-16,9,Athletico-PR,1,3.0
3226,2024-06-20,10,Bahia,3,1.0


In [20]:
historico_clubes["pontos_ultimos_5"] = (
    historico_clubes
    .groupby(["temporada", "clube"])["pontos_jogo_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).sum()
    )
)

In [21]:
historico_clubes[
    (historico_clubes["clube"] == "Flamengo") &
    (historico_clubes["temporada"] == 2024)
][
    [
        "data",
        "rodada",
        "adversario",
        "pontos",
        "pontos_jogo_anterior",
        "pontos_ultimos_5"
    ]
].head(10)

,data,rodada,adversario,pontos,pontos_jogo_anterior,pontos_ultimos_5
3053,2024-04-14,1,Atletico-GO,3,NaN,NaN
3074,2024-04-17,2,Sao Paulo,3,3.0,3.0
3087,2024-04-21,3,Palmeiras,1,3.0,6.0
3102,2024-04-28,4,Botafogo-RJ,0,1.0,7.0
3119,2024-05-04,5,Bragantino,1,0.0,7.0
3130,2024-05-11,6,Corinthians,3,1.0,8.0
3155,2024-06-02,7,Vasco,3,3.0,8.0
3184,2024-06-13,8,Gremio,3,3.0,8.0
3199,2024-06-16,9,Athletico-PR,1,3.0,10.0
3226,2024-06-20,10,Bahia,3,1.0,11.0


In [22]:
historico_clubes["gols_pro_anterior"] = (
    historico_clubes
    .groupby(["temporada", "clube"])["gols_pro"]
    .shift(1)
)

In [23]:
historico_clubes["gols_contra_anterior"] = (
    historico_clubes
    .groupby(["temporada", "clube"])["gols_contra"]
    .shift(1)
)

In [24]:
historico_clubes["gols_pro_ultimos_5"] = (
    historico_clubes
    .groupby(["temporada", "clube"])["gols_pro_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).sum()
    )
)

In [25]:
historico_clubes["gols_contra_ultimos_5"] = (
    historico_clubes
    .groupby(["temporada", "clube"])["gols_contra_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).sum()
    )
)

In [26]:
historico_clubes[
    (historico_clubes["clube"] == "Flamengo") &
    (historico_clubes["temporada"] == 2024)
][
    [
        "data",
        "rodada",
        "adversario",
        "gols_pro",
        "gols_contra",
        "pontos_ultimos_5",
        "gols_pro_ultimos_5",
        "gols_contra_ultimos_5"
    ]
].head(10)

,data,rodada,adversario,gols_pro,gols_contra,pontos_ultimos_5,gols_pro_ultimos_5,gols_contra_ultimos_5
3053,2024-04-14,1,Atletico-GO,2,1,NaN,NaN,NaN
3074,2024-04-17,2,Sao Paulo,2,1,3.0,2.0,1.0
3087,2024-04-21,3,Palmeiras,0,0,6.0,4.0,2.0
3102,2024-04-28,4,Botafogo-RJ,0,2,7.0,4.0,2.0
3119,2024-05-04,5,Bragantino,1,1,7.0,4.0,4.0
3130,2024-05-11,6,Corinthians,2,0,8.0,5.0,5.0
3155,2024-06-02,7,Vasco,6,1,8.0,5.0,4.0
3184,2024-06-13,8,Gremio,2,1,8.0,9.0,4.0
3199,2024-06-16,9,Athletico-PR,1,1,10.0,11.0,5.0
3226,2024-06-20,10,Bahia,2,1,11.0,12.0,4.0


In [27]:
historico_clubes["jogos_anteriores_5"] = (
    historico_clubes
    .groupby(["temporada", "clube"])["pontos_jogo_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).count()
    )
)

In [28]:
historico_clubes["pontos_por_jogo_ultimos_5"] = (
    historico_clubes["pontos_ultimos_5"]
    / historico_clubes["jogos_anteriores_5"]
)

In [29]:
historico_clubes["media_gols_pro_ultimos_5"] = (
    historico_clubes["gols_pro_ultimos_5"]
    / historico_clubes["jogos_anteriores_5"]
)

In [30]:
historico_clubes["media_gols_contra_ultimos_5"] = (
    historico_clubes["gols_contra_ultimos_5"]
    / historico_clubes["jogos_anteriores_5"]
)

In [31]:
historico_clubes[
    (historico_clubes["clube"] == "Flamengo") &
    (historico_clubes["temporada"] == 2024)
][
    [
        "rodada",
        "pontos",
        "jogos_anteriores_5",
        "pontos_ultimos_5",
        "pontos_por_jogo_ultimos_5",
        "media_gols_pro_ultimos_5",
        "media_gols_contra_ultimos_5"
    ]
].head(10)

,rodada,pontos,jogos_anteriores_5,pontos_ultimos_5,pontos_por_jogo_ultimos_5,media_gols_pro_ultimos_5,media_gols_contra_ultimos_5
3053,1,3,0.0,NaN,NaN,NaN,NaN
3074,2,3,1.0,3.0,3.000000,2.000000,1.000000
3087,3,1,2.0,6.0,3.000000,2.000000,1.000000
3102,4,0,3.0,7.0,2.333333,1.333333,0.666667
3119,5,1,4.0,7.0,1.750000,1.000000,1.000000
3130,6,3,5.0,8.0,1.600000,1.000000,1.000000
3155,7,3,5.0,8.0,1.600000,1.000000,0.800000
3184,8,3,5.0,8.0,1.600000,1.800000,0.800000
3199,9,1,5.0,10.0,2.000000,2.200000,1.000000
3226,10,3,5.0,11.0,2.200000,2.400000,0.800000


In [32]:
historico_clubes["saldo_gols_ultimos_5"] = (
    historico_clubes["gols_pro_ultimos_5"]
    - historico_clubes["gols_contra_ultimos_5"]
)

In [33]:
historico_clubes["media_saldo_gols_ultimos_5"] = (
    historico_clubes["media_gols_pro_ultimos_5"]
    - historico_clubes["media_gols_contra_ultimos_5"]
)

In [34]:
historico_clubes["pontos_mesmo_mando_anterior"] = (
    historico_clubes
    .groupby(["temporada", "clube", "mando"])["pontos"]
    .shift(1)
)

In [35]:
historico_clubes["jogos_mesmo_mando_anteriores_5"] = (
    historico_clubes
    .groupby(["temporada", "clube", "mando"])["pontos_mesmo_mando_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).count()
    )
)

In [36]:
historico_clubes["pontos_mesmo_mando_ultimos_5"] = (
    historico_clubes
    .groupby(["temporada", "clube", "mando"])["pontos_mesmo_mando_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).sum()
    )
)

In [37]:
historico_clubes["pontos_por_jogo_mesmo_mando_ultimos_5"] = (
    historico_clubes["pontos_mesmo_mando_ultimos_5"]
    / historico_clubes["jogos_mesmo_mando_anteriores_5"]
)

In [38]:
historico_clubes[
    (historico_clubes["clube"] == "Fortaleza") &
    (historico_clubes["temporada"] == 2024) &
    (historico_clubes["mando"] == "casa")
][
    [
        "data",
        "rodada",
        "adversario",
        "pontos",
        "jogos_mesmo_mando_anteriores_5",
        "pontos_mesmo_mando_ultimos_5",
        "pontos_por_jogo_mesmo_mando_ultimos_5"
    ]
].head(10)

,data,rodada,adversario,pontos,jogos_mesmo_mando_anteriores_5,pontos_mesmo_mando_ultimos_5,pontos_por_jogo_mesmo_mando_ultimos_5
3066,2024-04-17,2,Cruzeiro,1,0.0,NaN,NaN
3108,2024-04-28,4,Bragantino,1,1.0,1.0,1.0
3136,2024-05-12,6,Botafogo-RJ,1,2.0,2.0,1.0
3162,2024-06-02,7,Athletico-PR,3,3.0,3.0,1.0
3218,2024-06-19,10,Gremio,3,4.0,6.0,1.5
3264,2024-06-26,12,Palmeiras,3,5.0,9.0,1.8
3280,2024-06-30,13,Juventude,3,5.0,11.0,2.2
3314,2024-07-07,15,Fluminense,3,5.0,13.0,2.6
3360,2024-07-17,17,Vitoria,3,5.0,15.0,3.0
3378,2024-07-21,18,Atletico-GO,3,5.0,15.0,3.0


In [39]:
historico_clubes["gols_pro_mesmo_mando_anterior"] = (
    historico_clubes
    .groupby(["temporada", "clube", "mando"])["gols_pro"]
    .shift(1)
)

historico_clubes["gols_contra_mesmo_mando_anterior"] = (
    historico_clubes
    .groupby(["temporada", "clube", "mando"])["gols_contra"]
    .shift(1)
)

In [40]:
historico_clubes["gols_pro_mesmo_mando_ultimos_5"] = (
    historico_clubes
    .groupby(["temporada", "clube", "mando"])["gols_pro_mesmo_mando_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).sum()
    )
)

In [41]:
historico_clubes["gols_contra_mesmo_mando_ultimos_5"] = (
    historico_clubes
    .groupby(["temporada", "clube", "mando"])["gols_contra_mesmo_mando_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).sum()
    )
)

In [42]:
historico_clubes["media_gols_pro_mesmo_mando_ultimos_5"] = (
    historico_clubes["gols_pro_mesmo_mando_ultimos_5"]
    / historico_clubes["jogos_mesmo_mando_anteriores_5"]
)

historico_clubes["media_gols_contra_mesmo_mando_ultimos_5"] = (
    historico_clubes["gols_contra_mesmo_mando_ultimos_5"]
    / historico_clubes["jogos_mesmo_mando_anteriores_5"]
)

In [43]:
historico_clubes["media_saldo_mesmo_mando_ultimos_5"] = (
    historico_clubes["media_gols_pro_mesmo_mando_ultimos_5"]
    - historico_clubes["media_gols_contra_mesmo_mando_ultimos_5"]
)

In [44]:
fortaleza_casa = (
    historico_clubes[
        (historico_clubes["clube"] == "Fortaleza") &
        (historico_clubes["temporada"] == 2024) &
        (historico_clubes["mando"] == "casa")
    ]
    .sort_values(["data", "id"])
    .reset_index(drop=True)
)

# Vamos testar o 6º jogo em casa.
# Portanto, já existem exatamente 5 jogos anteriores.
i = 5

jogo_atual = fortaleza_casa.iloc[i]

ultimos_5 = fortaleza_casa.iloc[i-5:i]

print("PARTIDA ATUAL")
print(
    jogo_atual[
        ["data", "rodada", "adversario", "gols_pro", "gols_contra", "pontos"]
    ]
)

print("\n5 JOGOS ANTERIORES")
print(
    ultimos_5[
        ["data", "adversario", "gols_pro", "gols_contra", "pontos"]
    ]
)

# Cálculo manual usando somente os jogos anteriores
pontos_manual = ultimos_5["pontos"].sum()
gols_pro_manual = ultimos_5["gols_pro"].sum()
gols_contra_manual = ultimos_5["gols_contra"].sum()

media_pontos_manual = pontos_manual / len(ultimos_5)
media_gols_pro_manual = gols_pro_manual / len(ultimos_5)
media_gols_contra_manual = gols_contra_manual / len(ultimos_5)

print("\nCOMPARAÇÃO")

print(
    "Pontos:",
    pontos_manual,
    "==",
    jogo_atual["pontos_mesmo_mando_ultimos_5"]
)

print(
    "Média pontos:",
    media_pontos_manual,
    "==",
    jogo_atual["pontos_por_jogo_mesmo_mando_ultimos_5"]
)

print(
    "Gols pró:",
    gols_pro_manual,
    "==",
    jogo_atual["gols_pro_mesmo_mando_ultimos_5"]
)

print(
    "Média gols pró:",
    media_gols_pro_manual,
    "==",
    jogo_atual["media_gols_pro_mesmo_mando_ultimos_5"]
)

print(
    "Gols contra:",
    gols_contra_manual,
    "==",
    jogo_atual["gols_contra_mesmo_mando_ultimos_5"]
)

print(
    "Média gols contra:",
    media_gols_contra_manual,
    "==",
    jogo_atual["media_gols_contra_mesmo_mando_ultimos_5"]
)

PARTIDA ATUAL
data           2024-06-26 00:00:00
rodada                          12
adversario               Palmeiras
gols_pro                         3
gols_contra                      0
pontos                           3
Name: 5, dtype: object

5 JOGOS ANTERIORES
        data    adversario  gols_pro  gols_contra  pontos
0 2024-04-17      Cruzeiro         1            1       1
1 2024-04-28    Bragantino         1            1       1
2 2024-05-12   Botafogo-RJ         1            1       1
3 2024-06-02  Athletico-PR         1            0       3
4 2024-06-19        Gremio         1            0       3

COMPARAÇÃO
Pontos: 9 == 9.0
Média pontos: 1.8 == 1.8
Gols pró: 5 == 5.0
Média gols pró: 1.0 == 1.0
Gols contra: 3 == 3.0
Média gols contra: 0.6 == 0.6


In [45]:
assert pontos_manual == jogo_atual["pontos_mesmo_mando_ultimos_5"]

assert abs(
    media_pontos_manual
    - jogo_atual["pontos_por_jogo_mesmo_mando_ultimos_5"]
) < 1e-9

assert gols_pro_manual == jogo_atual["gols_pro_mesmo_mando_ultimos_5"]

assert abs(
    media_gols_pro_manual
    - jogo_atual["media_gols_pro_mesmo_mando_ultimos_5"]
) < 1e-9

assert gols_contra_manual == jogo_atual["gols_contra_mesmo_mando_ultimos_5"]

assert abs(
    media_gols_contra_manual
    - jogo_atual["media_gols_contra_mesmo_mando_ultimos_5"]
) < 1e-9

print("Todas as features conferem.")

Todas as features conferem.


## Conferência da ordem cronológica

O exemplo acima confere jogos no mesmo mando. A checagem abaixo confere os
cinco jogos anteriores do clube considerando casa e fora, para detectar
janelas que teriam sido calculadas com os dois blocos fora de ordem.

In [46]:
flamengo_2024 = historico_clubes[
    (historico_clubes['clube'] == 'Flamengo')
    & (historico_clubes['temporada'] == 2024)
].reset_index(drop=True)
jogo_atual_geral = flamengo_2024.iloc[5]
cinco_anteriores = flamengo_2024.iloc[:5]

assert (cinco_anteriores['data'] < jogo_atual_geral['data']).all()
assert cinco_anteriores['pontos'].sum() == jogo_atual_geral['pontos_ultimos_5']
assert cinco_anteriores['gols_pro'].sum() == jogo_atual_geral['gols_pro_ultimos_5']
assert cinco_anteriores['gols_contra'].sum() == jogo_atual_geral['gols_contra_ultimos_5']
print('Histórico geral em ordem cronológica conferido.')

Histórico geral em ordem cronológica conferido.


In [47]:
features_base = [
    "pontos_por_jogo_ultimos_5",
    "media_gols_pro_ultimos_5",
    "media_gols_contra_ultimos_5",
    "pontos_por_jogo_mesmo_mando_ultimos_5",
    "media_gols_pro_mesmo_mando_ultimos_5",
    "media_gols_contra_mesmo_mando_ultimos_5",
]

In [48]:
features_mandante = historico_clubes[
    historico_clubes["mando"] == "casa"
][
    ["id"] + features_base
].copy()

In [49]:
features_mandante = features_mandante.rename(
    columns={
        feature: f"mandante_{feature}"
        for feature in features_base
    }
)

## Uma linha por partida

Selecionamos as mesmas seis métricas para mandante e visitante e as
juntamos ao jogo pelo ID. `resultado` é o alvo futuro; gols da própria
partida não entram nas features. A junção exige um registro de cada
lado para cada jogo.

In [50]:
features_visitante = historico_clubes[
    historico_clubes['mando'] == 'fora'
][['id'] + features_base].copy()
features_visitante = features_visitante.rename(
    columns={feature: f'visitante_{feature}' for feature in features_base}
)

In [51]:
base_features = (
    df[['id', 'temporada', 'rodada', 'data', 'mandante', 'visitante', 'resultado']]
    .merge(features_mandante, on='id', validate='one_to_one')
    .merge(features_visitante, on='id', validate='one_to_one')
    .sort_values(['temporada', 'data', 'id'])
    .reset_index(drop=True)
)
assert len(base_features) == len(df) == 1900
assert base_features['id'].is_unique
base_features.shape

(1900, 19)

In [52]:
base_features.head(10)

,id,temporada,rodada,data,mandante,visitante,resultado,mandante_pontos_por_jogo_ultimos_5,mandante_media_gols_pro_ultimos_5,mandante_media_gols_contra_ultimos_5,mandante_pontos_por_jogo_mesmo_mando_ultimos_5,mandante_media_gols_pro_mesmo_mando_ultimos_5,mandante_media_gols_contra_mesmo_mando_ultimos_5,visitante_pontos_por_jogo_ultimos_5,visitante_media_gols_pro_ultimos_5,visitante_media_gols_contra_ultimos_5,visitante_pontos_por_jogo_mesmo_mando_ultimos_5,visitante_media_gols_pro_mesmo_mando_ultimos_5,visitante_media_gols_contra_mesmo_mando_ultimos_5
0,6886,2020,1,2020-08-08,Fortaleza,Athletico-PR,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6887,2020,1,2020-08-08,Coritiba,Internacional,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6888,2020,1,2020-08-08,Sport,Ceara,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6889,2020,1,2020-08-09,Santos,Bragantino,D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6890,2020,1,2020-08-09,Flamengo,Atletico-MG,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6891,2020,1,2020-08-09,Gremio,Fluminense,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6896,2020,2,2020-08-12,Athletico-PR,Goias,H,3.0,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,6897,2020,2,2020-08-12,Atletico-MG,Corinthians,H,3.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,6898,2020,2,2020-08-12,Bragantino,Botafogo-RJ,D,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,6899,2020,2,2020-08-12,Bahia,Coritiba,H,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.0,NaN,NaN,NaN


In [53]:
colunas_features = [
    f'{lado}_{feature}'
    for lado in ('mandante', 'visitante')
    for feature in features_base
]
base_features[colunas_features].isna().sum()

mandante_pontos_por_jogo_ultimos_5                    50
mandante_media_gols_pro_ultimos_5                     50
mandante_media_gols_contra_ultimos_5                  50
mandante_pontos_por_jogo_mesmo_mando_ultimos_5       100
mandante_media_gols_pro_mesmo_mando_ultimos_5        100
mandante_media_gols_contra_mesmo_mando_ultimos_5     100
visitante_pontos_por_jogo_ultimos_5                   50
visitante_media_gols_pro_ultimos_5                    50
visitante_media_gols_contra_ultimos_5                 50
visitante_pontos_por_jogo_mesmo_mando_ultimos_5      100
visitante_media_gols_pro_mesmo_mando_ultimos_5       100
visitante_media_gols_contra_mesmo_mando_ultimos_5    100
dtype: int64

In [54]:
caminho_features = Path('../data/processed/matches_features_2020_2024.csv')
base_features.to_csv(caminho_features, index=False)
print(f'Base de features salva em: {caminho_features}')
print(f'Partidas: {len(base_features)} | Features: {len(colunas_features)}')

Base de features salva em: ../data/processed/matches_features_2020_2024.csv
Partidas: 1900 | Features: 12
